# Lesson 15: Attention Intuition

Run each cell with Shift+Enter and read the output. This notebook builds the
intuition for attention by hand: a soft weighted-average lookup. No training here,
just the core idea. The exercises are at the bottom.

In [ ]:
import torch
import torch.nn as nn

## Attention is a soft lookup

Give each word a small "value" vector, choose how much to attend to each word
(the weights), and take the weighted average. The result is dominated by whichever
word gets the most weight. Here the weights are hardcoded; in a real model they are
computed from the input.

In [ ]:
words  = ["the", "cat", "sat", "down"]
values = torch.tensor([[0.1, 0.2],   # the
                       [0.8, 0.9],   # cat
                       [0.3, 0.1],   # sat
                       [0.2, 0.3]])  # down

weights = torch.tensor([0.05, 0.80, 0.10, 0.05])   # 80% on 'cat'
result  = weights @ values
print("result:", result)   # close to the "cat" vector

## Softmax turns raw scores into weights

The raw similarity scores can be any real numbers. `softmax` turns them into weights
that are all positive and sum to 1, so they are a valid weighted average. The biggest
score gets the biggest weight.

In [ ]:
scores  = torch.tensor([1.0, 3.5, 1.2, 0.8])
weights = torch.softmax(scores, dim=0)
print("weights:", weights)
print("sum:    ", weights.sum())   # 1.0

## The attention matrix

Do this for every token at once and you get a square matrix. Row i is where token i
is looking, and every row sums to 1.

In [ ]:
sentence  = ["The", "cat", "sat"]
attention = torch.tensor([[0.8, 0.1, 0.1],   # 'The' -> itself
                          [0.2, 0.6, 0.2],   # 'cat' -> itself
                          [0.1, 0.5, 0.4]])  # 'sat' -> 'cat'
print("row sums:", attention.sum(dim=1))   # all 1.0

## Shapes: content changes, shape does not

Project the input to Q, K, V, score every token against every token, softmax the
scores, and average the values. The output has the same shape as the input, which is
why we can stack attention layers. (We skip the sqrt(d) scaling here; that arrives in
Lesson 17.)

In [ ]:
x = torch.randn(2, 5, 8)             # (batch, seq, embed_dim)
Wq, Wk, Wv = nn.Linear(8, 8), nn.Linear(8, 8), nn.Linear(8, 8)
Q, K, V = Wq(x), Wk(x), Wv(x)

scores  = Q @ K.transpose(-2, -1)   # (2, 5, 5)
weights = torch.softmax(scores, dim=-1)
out     = weights @ V               # (2, 5, 8)
print("scores:", scores.shape)
print(x.shape, "->", out.shape)     # same shape

## Exercises

Fill in each cell under the `YOUR CODE HERE` line and run it. Print shapes and values
to check your own work. When you are done, upload this notebook on the lesson page to
get feedback and a grade.

In [ ]:
# E1: the four value vectors below are given. Build a weights vector that puts
#     most of the attention on 'down' (the last word), make sure it sums to 1,
#     and print the weighted-average result.
values = torch.tensor([[0.1, 0.2],
                       [0.8, 0.9],
                       [0.3, 0.1],
                       [0.2, 0.3]])
# YOUR CODE HERE (write your answer below)


In [ ]:
# E2: apply softmax to these raw scores, then print the weights and their sum.
#     Which word gets the most weight, and why?
scores = torch.tensor([2.0, 2.0, 5.0])
# YOUR CODE HERE (write your answer below)


In [ ]:
# E3: make a random input x of shape (batch=2, seq=6, embed_dim=16). Project it to
#     Q, K, V with three nn.Linear(16, 16), compute the attention output, and print
#     x.shape and out.shape. Confirm they are equal.
# YOUR CODE HERE (write your answer below)


In [ ]:
# E4: write a 3x3 attention matrix for the sentence ['I', 'like', 'cats'] where the
#     last word 'cats' attends mostly to itself but also looks back at 'like'.
#     Print the row sums and check that every row sums to 1.
# YOUR CODE HERE (write your answer below)
